# Frigate / SPOT / local comparator — conditioned image consistency

Compare the production SPOT optics and detector with a local Gaussian-filtering and Poisson-sampling comparator. Run top to bottom from the `python/` directory. Optional observation files are required in `Inputs/Validation/`; see that directory's README.

**Scope:** the background, manual trails, amplitudes, effective gain and PSF are derived from or selected for the measured image. Orbit, visibility and target BRDF prediction are outside this detector-plane comparison. The default `internal_satsim_like` mode is labelled **SatSim-like** and does not execute the SatSim package.

The optional `external_satsim` mode reads registered SatSim images and their provenance; it does not launch SatSim. For direct execution of native SatSim optical/noise routines, use `03_Validation_Native_SatSim.ipynb`.


## 1. Imports and editable paths

In [ ]:
import copy, hashlib, json, sys
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import binary_dilation, gaussian_filter

ROOT = Path.cwd().resolve()
if not (ROOT / 'ssa_emulator').is_dir():
    raise RuntimeError('Start Jupyter in the python folder containing ssa_emulator.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import ssa_emulator
from ssa_emulator.config import default_config
from ssa_emulator.frigate_conditioning import (
    apply_spot_to_conditioned_maps, background_residual,
    configure_spot_for_detector_maps, downsample_image,
    extract_frigate_metadata, extract_wcs_metadata, load_fits_image,
    load_solved_wcs, manual_trails_from_roi, project_catalogue_to_detector,
    render_star_flux, render_trail_flux, scale_roi, load_validation_catalogue,
)
from ssa_emulator.validation import (
    align_background_median, ssim_metric, psnr_metric, direct_pair_metrics,
    target_integrated_flux, frigate_reference_metrics, load_external_satsim,
)
from ssa_emulator.export import json_safe

DATA_DIR = ROOT / 'Inputs' / 'Validation'
RAW_FITS = DATA_DIR / 'Capture_00938 03_05_42Z.fits'
WCS_FITS = DATA_DIR / 'wcs.fits'
SOLVED_FITS = DATA_DIR / 'new-image.fits'
CATALOGUE = DATA_DIR / 'StarDB_500k.mat'  # exact catalogue; normal examples use MATLAB's CSV
OUT_DIR = ROOT / 'Outputs' / ('validation_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
OUT_DIR.mkdir(parents=True, exist_ok=False)
DOWNSAMPLE = 4
ROI_FULL = (6300, 5440, 7300, 6260)
ROI_WORK = scale_roi(ROI_FULL, DOWNSAMPLE)

COMPARATOR_MODE = 'internal_satsim_like'  # or 'external_satsim'
SATSIM_NOISE_OFF_IMAGE_PATH = None
SATSIM_NOISE_ON_IMAGE_PATH = None
SATSIM_NOISE_OFF_METADATA_PATH = None
SATSIM_NOISE_ON_METADATA_PATH = None
print('Engine:', Path(ssa_emulator.__file__).resolve())
print('Results:', OUT_DIR)

## 2. Load measured FITS/WCS and the six measured trail signatures

The validation uses block means and discards the final incomplete block: the 6422 × 9600 input becomes 1605 × 2400. Endpoints are zero-based pixels in the 205 × 250 working ROI. The coordinate convention is preserved.

In [ ]:
# Load FITS & WCS
frigate_full, raw_header = load_fits_image(RAW_FITS)
frigate = downsample_image(frigate_full, DOWNSAMPLE)
meta = extract_frigate_metadata(RAW_FITS, frigate_full, raw_header)
wcs_path = WCS_FITS if WCS_FITS.exists() else SOLVED_FITS
wcs = load_solved_wcs(wcs_path)
wcs_meta = extract_wcs_metadata(wcs, frigate_full.shape)

x0, y0, x1, y1 = ROI_WORK
frigate_roi = frigate[y0:y1, x0:x1]

# Trail signatures (Local ROI endpoints)
manual_trails_roi = pd.DataFrame(
    [
        [1, 55, 156, 59, 128, 4.0],
        [2, 65, 160, 71, 137, 4.0],
        [3, 84, 165, 88, 141, 4.0],
        [4, 96, 154, 101, 128, 4.0],
        [5, 106, 147, 114, 118, 4.0],
        [6, 123, 132, 132, 103, 4.0],
    ],
    columns=["id", "x1_roi", "y1_roi", "x2_roi", "y2_roi", "width_px"],
)

trail_df = manual_trails_from_roi(
    manual_trails_roi,
    roi_full_xyxy=ROI_FULL,
    downsample=DOWNSAMPLE,
    wcs=wcs,
)

## 3. Project the validation catalogue

In [ ]:
p_base = default_config('ground', 'full')
stars = load_validation_catalogue(CATALOGUE, max_mag=13.5)
projected_stars = project_catalogue_to_detector(
    stars, wcs, image_shape=frigate_full.shape,
    max_mag=13.5, max_stars=None, downsample=DOWNSAMPLE,
)
stars_in_roi = projected_stars[
    (projected_stars['x'] >= x0) & (projected_stars['x'] < x1)
    & (projected_stars['y'] >= y0) & (projected_stars['y'] < y1)
].copy()
print(len(projected_stars), 'full-frame stars;', len(stars_in_roi), 'ROI stars')

## 4. Condition the scene 

Full-frame 101-pixel median background, MAD noise estimate, percentile amplitude fits, star boost 28, trail boost 1.5, trail smoothing 1.3 pixels, relative support threshold 0.001 and three dilations. The full-frame median filter can take several minutes.

In [ ]:
# Frigate Conditioning Scene Generation
bg, resid, norm_resid, bg_sigma = background_residual(
    frigate, filter_size=101
)

star_unit = render_star_flux(
    frigate.shape, projected_stars, zero_mag_flux=1.0, psf_sigma_px=0.0
)

trail_unit = render_trail_flux(
    frigate.shape, trail_df, amplitude=1.0, sigma_px=1.3
)

star_roi_unit = star_unit[y0:y1, x0:x1]
trail_roi_unit = trail_unit[y0:y1, x0:x1]
resid_roi = resid[y0:y1, x0:x1]

star_nonzero = star_roi_unit[star_roi_unit > 0]
trail_nonzero = trail_roi_unit[trail_roi_unit > 0]
positive_resid = resid_roi[resid_roi > np.percentile(resid_roi, 90)]

STAR_SCALE = 1.0
TRAIL_SCALE = 1.0

if len(star_nonzero) > 0 and len(positive_resid) > 0:
    STAR_SCALE = np.percentile(positive_resid, 70) / max(
        np.percentile(star_nonzero, 99.5), 1e-12
    )

if len(trail_nonzero) > 0:
    TRAIL_SCALE = np.percentile(resid_roi, 99.8) / max(
        np.percentile(trail_nonzero, 99.9), 1e-12
    )

STAR_BOOST = 28.0
TRAIL_BOOST = 1.5
STAR_SCALE *= STAR_BOOST
TRAIL_SCALE *= TRAIL_BOOST

image_stars_flux = (STAR_SCALE * star_unit).astype(np.float32)
image_targets_flux = (TRAIL_SCALE * trail_unit).astype(np.float32)
image_point_flux = (image_stars_flux + image_targets_flux).astype(np.float32)
image_background_flux = bg.astype(np.float32)

target_peak = float(np.nanmax(image_targets_flux))
target_support = image_targets_flux >= max(target_peak * 1e-3, 1e-12)
target_support = binary_dilation(target_support, iterations=3)

## 5. Run SPOT and the local comparator

SPOT uses seed 12 and the comparator uses seed 21. Noise is Poisson only; gain is median background / robust variance. PSF sigma remains 0.75 working pixels. The background is added after the point-source PSF.

In [ ]:
# Calibrated Rendering Wrappers
fitted_psf_sigma = 0.75


def run_spot_case(noise_on, random_seed):
    bg_mean_dn = float(np.nanmedian(image_background_flux))
    p = configure_spot_for_detector_maps(
        copy.deepcopy(p_base),
        image_shape=frigate.shape,
        exposure_s=meta.exposure_s or 0.5,
        fov_x_deg=wcs_meta.fov_x_deg,
        fov_y_deg=wcs_meta.fov_y_deg,
        deterministic=not noise_on,
        random_seed=random_seed,
        bg_mean_dn=bg_mean_dn,
        bg_sigma_dn=bg_sigma,
        psf_sigma=fitted_psf_sigma,
    )

    result = apply_spot_to_conditioned_maps(
        p,
        point_source_flux=image_point_flux,
        diffuse_background_flux=image_background_flux,
        exposure_s=meta.exposure_s or 0.5,
    )

    return np.asarray(result["Image"], dtype=np.float32), result


def run_satsim_like_case(noise_on, random_seed):
    rng = np.random.default_rng(int(random_seed))
    point = np.asarray(image_point_flux, dtype=np.float64)
    background = np.asarray(image_background_flux, dtype=np.float64)

    point_after_psf = gaussian_filter(
        point, sigma=fitted_psf_sigma, mode="nearest"
    )
    signal_mean_dn = np.maximum(background + point_after_psf, 0.0)

    bg_mean_dn = float(np.nanmedian(image_background_flux))
    gain = float(bg_mean_dn / max(bg_sigma**2, 1e-6))

    if noise_on:
        signal_e = signal_mean_dn * gain
        noisy_e = rng.poisson(signal_e).astype(np.float64)
        image = noisy_e / gain
    else:
        image = signal_mean_dn.copy()

    full_well = float(np.nanmax(frigate) * 3.0)
    return np.clip(image, 0.0, full_well).astype(np.float32)


spot_noise_off, _ = run_spot_case(noise_on=False, random_seed=12)
spot_noise_on, _ = run_spot_case(noise_on=True, random_seed=12)

## 6. Select the comparator arrays

In [ ]:
satsim_provenance = {}
if COMPARATOR_MODE == 'internal_satsim_like':
    satsim_noise_off = run_satsim_like_case(noise_on=False, random_seed=21)
    satsim_noise_on = run_satsim_like_case(noise_on=True, random_seed=21)
    COMPARATOR_NAME = 'SatSim-like'
    SATSIM_SOURCE = 'local Gaussian/Poisson comparator; SatSim was not executed'
elif COMPARATOR_MODE == 'external_satsim':
    satsim_noise_off, satsim_provenance['noise_off'] = load_external_satsim(
        SATSIM_NOISE_OFF_IMAGE_PATH, SATSIM_NOISE_OFF_METADATA_PATH, frigate.shape, False)
    satsim_noise_on, satsim_provenance['noise_on'] = load_external_satsim(
        SATSIM_NOISE_ON_IMAGE_PATH, SATSIM_NOISE_ON_METADATA_PATH, frigate.shape, True)
    COMPARATOR_NAME = 'SatSim (external)'
    SATSIM_SOURCE = 'user-supplied registered raw SatSim outputs with provenance'
else:
    raise ValueError('COMPARATOR_MODE must be internal_satsim_like or external_satsim')
satsim_figure8 = satsim_noise_on
spot_figure8 = spot_noise_on
FIGURE8_COMPARATOR_NAME = COMPARATOR_NAME
print(SATSIM_SOURCE)

## 7. Table IV — noise-off and noise-on image consistency

Direct (unaligned) SSIM, PSNR, RMSE and MAE on the fixed ROI. Target flux is the sum of positive background-subtracted pixels in the support mask. The background is the median outside that support.

In [ ]:
# ============================================================
# 14. Generate Table IV from the Frigate-conditioned scene
# ============================================================

def table_iv_case(spot_image, comparator_image):
    x0, y0, x1, y1 = ROI_WORK

    spot_crop = np.asarray(
        spot_image[y0:y1, x0:x1],
        dtype=np.float64,
    )

    comparator_crop = np.asarray(
        comparator_image[y0:y1, x0:x1],
        dtype=np.float64,
    )

    support_crop = target_support[y0:y1, x0:x1]

    pair = direct_pair_metrics(
        comparator_crop,
        spot_crop,
    )

    comparator_flux, comparator_bg = target_integrated_flux(
        comparator_crop,
        support_crop,
    )

    spot_flux, spot_bg = target_integrated_flux(
        spot_crop,
        support_crop,
    )

    flux_difference_percent = (
        100.0
        * abs(spot_flux - comparator_flux)
        / max(abs(comparator_flux), 1e-12)
    )

    return {
        **pair,
        "Comparator_background_median_DN": comparator_bg,
        "SPOT_background_median_DN": spot_bg,
        "Comparator_target_flux_DN": comparator_flux,
        "SPOT_target_flux_DN": spot_flux,
        "Target_flux_difference_percent": flux_difference_percent,
    }

table_iv_raw = pd.DataFrame(
    {
        "Noise-off": table_iv_case(
            spot_noise_off,
            satsim_noise_off,
        ),
        "Noise-on": table_iv_case(
            spot_noise_on,
            satsim_noise_on,
        ),
    }
).T

display(table_iv_raw)

no = table_iv_raw.loc["Noise-off"]
on = table_iv_raw.loc["Noise-on"]

table_iv = pd.DataFrame(
    {
        "Metric": [
            "SSIM",
            "PSNR [dB]",
            "RMSE [DN]",
            "MAE [DN]",
            f"{COMPARATOR_NAME} background median [DN]",
            "SPOT background median [DN]",
            f"{COMPARATOR_NAME} target flux [DN]",
            "SPOT target flux [DN]",
            "Target flux difference [%]",
        ],
        "Noise-off": [
            f"{no['SSIM']:.4f}",
            f"{no['PSNR_dB']:.2f}",
            f"{no['RMSE_DN']:.2f}",
            f"{no['MAE_DN']:.2f}",
            f"{no['Comparator_background_median_DN']:.2f}",
            f"{no['SPOT_background_median_DN']:.2f}",
            f"{no['Comparator_target_flux_DN']:.2f}",
            f"{no['SPOT_target_flux_DN']:.2f}",
            f"{no['Target_flux_difference_percent']:.2f}",
        ],
        "Noise-on": [
            f"{on['SSIM']:.4f}",
            f"{on['PSNR_dB']:.2f}",
            f"{on['RMSE_DN']:.2f}",
            f"{on['MAE_DN']:.2f}",
            f"{on['Comparator_background_median_DN']:.2f}",
            f"{on['SPOT_background_median_DN']:.2f}",
            f"{on['Comparator_target_flux_DN']:.2f}",
            f"{on['SPOT_target_flux_DN']:.2f}",
            f"{on['Target_flux_difference_percent']:.2f}",
        ],
    }
)

display(table_iv)

table_iv_path = OUT_DIR / "Table_IV_Frigate_conditioned_SPOT_vs_comparator.csv"
table_iv.to_csv(
    table_iv_path,
    index=False,
)

table_iv_raw_path = OUT_DIR / "Table_IV_audit_raw_metrics.csv"
table_iv_raw.to_csv(
    table_iv_raw_path,
)

print("Saved:", table_iv_path)
print("Saved:", table_iv_raw_path)

## 8. Figure 8 — the same three noise-on arrays

Each top panel uses its own [1, 99.8] percentiles. Residual panels use the 51-pixel median filter and fixed −3 to +8 normalization limits.

In [ ]:
# Plot Figure 8
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
raw_images = [frigate, satsim_figure8, spot_figure8]
raw_titles = [
    "Frigate FITS ROI (real data)",
    f"{FIGURE8_COMPARATOR_NAME} matched ROI",
    "SPOT matched ROI",
]

for ax, image, title, label in zip(
    axs[0], raw_images, raw_titles, ["(a)", "(b)", "(c)"]
):
    crop = image[y0:y1, x0:x1]
    vmin, vmax = np.percentile(crop, [1, 99.8])
    ax.imshow(
        crop, cmap="gray", origin="upper", vmin=vmin, vmax=vmax
    )
    ax.set_title(title)
    ax.text(
        0.03,
        0.95,
        label,
        transform=ax.transAxes,
        va="top",
        ha="left",
        color="white",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xticks([])
    ax.set_ylabel("y pixel")
    ax.tick_params(axis="y", which="both", left=True, labelleft=True)

figure8_sigmas = {}
residual_titles = [
    "Frigate normalized residual (real data)",
    f"{FIGURE8_COMPARATOR_NAME} normalized residual",
    "SPOT normalized residual",
]

for ax, image, title, label, key in zip(
    axs[1],
    raw_images,
    residual_titles,
    ["(d)", "(e)", "(f)"],
    ["Frigate", FIGURE8_COMPARATOR_NAME, "SPOT"],
):
    crop = image[y0:y1, x0:x1]
    _, _, norm, sigma = background_residual(crop, filter_size=51)
    figure8_sigmas[key] = sigma
    ax.imshow(norm, cmap="gray", origin="upper", vmin=-3, vmax=8)
    ax.set_title(f"{title}, σ={sigma:.2f}")
    ax.text(
        0.03,
        0.95,
        label,
        transform=ax.transAxes,
        va="top",
        ha="left",
        color="white",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")

plt.tight_layout()
plt.savefig(
    OUT_DIR / "Figure8_Frigate_comparator_SPOT_scene_matched.png",
    dpi=400,
    bbox_inches="tight",
)
plt.show()

## 9. Table V — comparison to the measured Frigate frame

Only a scalar median offset is applied for aligned errors. Raw statistics remain unaltered.

In [ ]:
# ============================================================
# 16. Generate Table V from the SAME Figure 8 arrays
# ============================================================

frigate_crop = np.asarray(
    frigate[y0:y1, x0:x1],
    dtype=np.float64,
)

comparator_crop = np.asarray(
    satsim_figure8[y0:y1, x0:x1],
    dtype=np.float64,
)

spot_crop = np.asarray(
    spot_figure8[y0:y1, x0:x1],
    dtype=np.float64,
)

frigate_stats = {
    "median_dn": float(np.nanmedian(frigate_crop)),
    "p99_dn": float(np.nanpercentile(frigate_crop, 99)),
    "residual_sigma_dn": figure8_sigmas["Frigate"],
}

comparator_stats = frigate_reference_metrics(
    frigate_crop,
    comparator_crop,
)

spot_stats = frigate_reference_metrics(
    frigate_crop,
    spot_crop,
)

roi_size = f"{x1 - x0} x {y1 - y0}"

table_v = pd.DataFrame(
    {
        "Metric": [
            "Number of frames",
            "Selected image size [pixels]",
            "Projected catalogue stars in full frame",
            "Projected catalogue stars in selected pixels",
            "Image-derived trail signatures",
            "Median [DN]",
            "Percentile p99 [DN]",
            "Robust residual noise [DN]",
            "Background-aligned RMSE to Frigate [DN]",
            "Background-aligned MAE to Frigate [DN]",
        ],
        "Frigate": [
            "1",
            roi_size,
            str(len(projected_stars)),
            str(len(stars_in_roi)),
            str(len(trail_df)),
            f"{frigate_stats['median_dn']:.2f}",
            f"{frigate_stats['p99_dn']:.2f}",
            f"{frigate_stats['residual_sigma_dn']:.2f}",
            "--",
            "--",
        ],
        FIGURE8_COMPARATOR_NAME: [
            "1",
            roi_size,
            str(len(projected_stars)),
            str(len(stars_in_roi)),
            str(len(trail_df)),
            f"{comparator_stats['median_dn']:.2f}",
            f"{comparator_stats['p99_dn']:.2f}",
            f"{comparator_stats['residual_sigma_dn']:.2f}",
            f"{comparator_stats['background_aligned_rmse_dn']:.2f}",
            f"{comparator_stats['background_aligned_mae_dn']:.2f}",
        ],
        "SPOT": [
            "1",
            roi_size,
            str(len(projected_stars)),
            str(len(stars_in_roi)),
            str(len(trail_df)),
            f"{spot_stats['median_dn']:.2f}",
            f"{spot_stats['p99_dn']:.2f}",
            f"{spot_stats['residual_sigma_dn']:.2f}",
            f"{spot_stats['background_aligned_rmse_dn']:.2f}",
            f"{spot_stats['background_aligned_mae_dn']:.2f}",
        ],
    }
)

display(table_v)

table_v_path = OUT_DIR / "Table_V_Frigate_conditioned_scene_matched.csv"

table_v.to_csv(
    table_v_path,
    index=False,
)

print("Saved:", table_v_path)

## 10. Supplementary audit statistics

In [ ]:
# ============================================================
# 17. Supplementary audit statistics
# ============================================================
# These are useful for checking the run, but are not required
# in the manuscript Table V.

audit_table = pd.DataFrame(
    {
        "Metric": [
            "SSIM to Frigate",
            "PSNR to Frigate [dB]",
            "Pearson r to Frigate",
            "Raw RMSE to Frigate [DN]",
            "Raw MAE to Frigate [DN]",
            "Residual-noise ratio to Frigate",
        ],
        FIGURE8_COMPARATOR_NAME: [
            f"{comparator_stats['ssim_to_frigate']:.4f}",
            f"{comparator_stats['psnr_to_frigate_db']:.2f}",
            f"{comparator_stats['pearson_r_to_frigate']:.4f}",
            f"{comparator_stats['raw_rmse_dn']:.2f}",
            f"{comparator_stats['raw_mae_dn']:.2f}",
            f"{comparator_stats['residual_sigma_ratio_to_frigate']:.3f}",
        ],
        "SPOT": [
            f"{spot_stats['ssim_to_frigate']:.4f}",
            f"{spot_stats['psnr_to_frigate_db']:.2f}",
            f"{spot_stats['pearson_r_to_frigate']:.4f}",
            f"{spot_stats['raw_rmse_dn']:.2f}",
            f"{spot_stats['raw_mae_dn']:.2f}",
            f"{spot_stats['residual_sigma_ratio_to_frigate']:.3f}",
        ],
    }
)

display(audit_table)

audit_path = OUT_DIR / "Table_V_audit_statistics.csv"

audit_table.to_csv(
    audit_path,
    index=False,
)

print("Saved:", audit_path)

## 11. Save numerical arrays and provenance

The saved arrays are the actual arrays used by both tables and Figure 8. No screenshot-derived or resized data enter the metrics.

In [ ]:
np.savez_compressed(
    OUT_DIR / 'validation_arrays.npz',
    Frigate=frigate, SPOT_noise_off=spot_noise_off, SPOT_noise_on=spot_noise_on,
    Comparator_noise_off=satsim_noise_off, Comparator_noise_on=satsim_noise_on,
    TargetSupport=target_support, PointInput=image_point_flux,
    BackgroundInput=image_background_flux,
)
projected_stars.to_csv(OUT_DIR / 'projected_stars.csv', index=False)
trail_df.to_csv(OUT_DIR / 'manual_trails.csv', index=False)
_, spot_configuration = run_spot_case(noise_on=False, random_seed=12)
provenance = dict(
    methodology='Supplied Frigate-conditioned scene-matched validation methodology',
    independent_full_forward_validation=False, fitted_and_evaluated_on_same_roi=True,
    comparator_mode=COMPARATOR_MODE, comparator_source=SATSIM_SOURCE,
    satsim_executed_here=False, external_satsim=satsim_provenance,
    roi_full=ROI_FULL, roi_work=ROI_WORK, downsample=DOWNSAMPLE,
    downsample_method='arithmetic block mean, incomplete edge blocks discarded',
    star_scale=STAR_SCALE, trail_scale=TRAIL_SCALE, star_boost=STAR_BOOST,
    trail_boost=TRAIL_BOOST, fitted_psf_sigma=fitted_psf_sigma, background_sigma=bg_sigma,
    spot_config=spot_configuration['Configuration'],
    backend_changes=['production SPOT optics finite Gaussian / zero boundary',
                     'production detector electron full-well and MT19937 streams'],
    input_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                  for p in (RAW_FITS, wcs_path, CATALOGUE)},
    source_sha256={str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
                   for p in sorted((ROOT / 'ssa_emulator').glob('*.py'))},
)
(OUT_DIR / 'provenance.json').write_text(
    json.dumps(json_safe(provenance), indent=2, allow_nan=False), encoding='utf-8')
print('Saved all validation products to', OUT_DIR)